# Baseline model
Week 4: a Linear Regression baseline that every later model has to beat by a margin that isn't noise (handbook §09).
CRISP-DM Modeling phase: test design written before training, then build, describe and assess the model.

Input: the cleaned splits from `02_preprocessing.ipynb` (`dataset/processed/`).

#### Setup

In [1]:
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# shared with 04 and 05: loading, Dataset A fills (fit on train only), encoding and metrics
from idx_pipeline import (CAT_COLS, DATA_DIR, LOG_COLS, SEED, TARGET, build_preprocessor, error_by,
                          load_split, metrics)

# SEED: LinearRegression is deterministic; fixed anyway for anything random added later (handbook §11)
np.random.seed(SEED)

MODEL_DIR = os.path.join(DATA_DIR, "models")  # inside dataset/, which .gitignore excludes
os.makedirs(MODEL_DIR, exist_ok=True)

#### Test design (written before training)
- **Split** (handbook §01): train = Jan 2024 – Feb 2026, validation = Mar 2026, test = Apr 2026. Chronological, no shuffling.
- **Choices** (which features, which variant) are made on **validation** only.
- **Test is not used here.** Per handbook §01 it's touched once, in Week 8, where the baseline and every later model are scored on it together. Loading it here would make it tempting to check after each model.
- **Metrics** (handbook §08): MdAPE is the headline; R², MAPE and MAE reported alongside.
- **Error breakdown**: by price band (quintiles of the actual price) and by county.
- **Success criterion**: the baseline has no target to hit; it sets the bar for Weeks 5–7. Later models must beat its validation MdAPE by a clear margin.

#### Step 1: Load the cleaned splits

In [2]:
# load_split is in idx_pipeline.py; PostalCode is already a 5-digit string from 02
data = {s: load_split(s) for s in ["train", "val"]}  # test is held out until Week 8
{s: d.shape for s, d in data.items()}

{'train': (282279, 42), 'val': (11412, 42)}

In [3]:
# Feature list comes from the dataset description made in 02, so roles are defined in one place.
# Excluded: id, target, comps_only (price_per_sqft contains the target; CloseDate, property_id are for comps).
description = pd.read_csv(os.path.join(DATA_DIR, "dataset_description.csv"))
features = description.loc[description["role"].isin(["model_feature", "missing_flag"]), "column"].tolist()

# CAT_COLS (City, CountyOrParish, PostalCode) and LOG_COLS (right-skewed sizes, log1p before scaling,
# see Step 4) come from idx_pipeline.py
NUM_COLS = [c for c in features if c not in CAT_COLS + LOG_COLS]
print(f"{len(LOG_COLS)} logged: {LOG_COLS}")
print(f"{len(NUM_COLS)} numeric: {NUM_COLS}")
print(f"{len(CAT_COLS)} categorical: {CAT_COLS}")

3 logged: ['LivingArea', 'LotSizeSquareFeet', 'GarageSpaces']
28 numeric: ['BedroomsTotal', 'BathroomsTotalInteger', 'YearBuilt', 'ParkingTotal', 'Stories', 'Latitude', 'Longitude', 'ViewYN', 'PoolPrivateYN', 'FireplaceYN', 'NewConstructionYN', 'AttachedGarageYN', 'AgeAtSale', 'month_sin', 'month_cos', 'LivingArea_missing', 'LotSizeSquareFeet_missing', 'GarageSpaces_missing', 'ParkingTotal_missing', 'YearBuilt_missing', 'BedroomsTotal_missing', 'BathroomsTotalInteger_missing', 'Stories_missing', 'ViewYN_missing', 'PoolPrivateYN_missing', 'FireplaceYN_missing', 'NewConstructionYN_missing', 'AttachedGarageYN_missing']
3 categorical: ['City', 'CountyOrParish', 'PostalCode']


#### Step 2: Metrics

In [4]:
# metrics() and error_by() live in idx_pipeline.py, so 03, 04 and 05 score models the same way.
# MdAPE is the headline (error on the typical home); MAPE, R², MAE and the share within 10% are reported alongside.
metrics(np.array([100.0, 200.0]), np.array([110.0, 180.0]))  # both homes 10% off

{'MdAPE_%': np.float64(10.0),
 'MAPE_%': np.float64(10.0),
 'R2': 0.9,
 'MAE_$': 15,
 'within_10%': np.float64(100.0)}

#### Step 3: Model 0, median price
A sanity floor: predict the train median price for every home. Any real model has to beat this easily.

In [5]:
train_median = data["train"][TARGET].median()
results = {"median_price": metrics(data["val"][TARGET].to_numpy(), np.full(len(data["val"]), train_median))}
results["median_price"]

{'MdAPE_%': np.float64(39.66),
 'MAPE_%': np.float64(49.87),
 'R2': -0.105,
 'MAE_$': 585657,
 'within_10%': np.float64(13.7)}

#### Step 4: Model 1, Linear Regression
- **Pipeline** (handbook §07, data-preprocessing.md "Important"): `build_preprocessor("linear")` from `idx_pipeline.py`, so `fit` only ever sees train:
  - `DatasetAImputer` fills the Dataset A missing values: train median for numbers, train mode for Y/N, 'Unknown' for City/county/ZIP, and lat/long by ZIP → county → overall median. The `_missing` flags come from 02.
  - then scaling and one-hot encoding in a `ColumnTransformer`.
- **Log target**: prices are right-skewed and errors scale with price, so the model predicts `log(ClosePrice)` and the prediction is converted back with `exp`. This fits percentage error, which is what MdAPE measures.
- **Log the skewed sizes** (LivingArea, LotSizeSquareFeet, GarageSpaces) with `log1p`. With a log target and raw sizes, each extra sq ft multiplies the price, so a 15,000 sqft home got a $100M+ prediction. Logged sizes make the effect "x% more area -> y% more price", which stays sensible at the extremes and also tames bad values like a 1.9 billion sqft lot. (AssociationFee is on hold until 05.)
- **Categoricals**: one-hot, with categories seen fewer than 30 times in train grouped as "infrequent". Unseen categories in val/test go to that group instead of erroring.
- **Two variants** to see how much location detail helps a linear model:
  - `lr_county`: property features + lat/long + county
  - `lr_zip`: the same + ZIP code (City is left out; ZIP is the finer version of the same information)

Target encoding for City/ZIP (handbook §05) is tried in 05.

In [6]:
def build_lr(cat_cols):
    pre = build_preprocessor("linear", NUM_COLS, cat_cols, LOG_COLS)  # impute -> log/scale/one-hot
    pipe = Pipeline([("pre", pre), ("lr", LinearRegression())])
    return TransformedTargetRegressor(regressor=pipe, func=np.log, inverse_func=np.exp)


variants = {
    "lr_county": ["CountyOrParish"],
    "lr_zip": ["CountyOrParish", "PostalCode"], #fix this
}
X = {s: d[features] for s, d in data.items()}
y = {s: d[TARGET].to_numpy() for s, d in data.items()}

models, val_pred = {}, {}
for name, cat_cols in variants.items():
    models[name] = build_lr(cat_cols).fit(X["train"], y["train"])  # TRAIN ONLY
    val_pred[name] = models[name].predict(X["val"])
    results[name] = metrics(y["val"], val_pred[name])

# train scores too: a big train/val gap would mean overfitting
for name in variants:
    results[f"{name} (train)"] = metrics(y["train"], models[name].predict(X["train"]))

val_results = pd.DataFrame(results).T
val_results

,MdAPE_%,MAPE_%,R2,MAE_$,within_10%
median_price,39.66,49.87,-0.1050,585657.0,13.7
lr_county,17.88,23.06,0.6310,303208.0,29.8
lr_zip,9.52,13.56,0.8298,181042.0,51.8
lr_county (train),17.72,23.04,0.6591,299459.0,30.0
lr_zip (train),9.29,13.07,0.8582,172747.0,53.1


In [7]:
# Check: the fitted pipeline leaves no missing values, on train or val
pre = models["lr_zip"].regressor_.named_steps["pre"]
for s in ["train", "val"]:
    out = pre.transform(X[s])
    out = out.toarray() if hasattr(out, "toarray") else out
    assert not np.isnan(out).any(), f"{s}: NaN after the pipeline"
    print(f"{s}: {out.shape[1]} model columns, no NaN")

train: 1004 model columns, no NaN
val: 1004 model columns, no NaN


In [8]:
# pick the variant with the lowest validation MdAPE
chosen = min(variants, key=lambda n: results[n]["MdAPE_%"])
print("chosen on validation:", chosen)

chosen on validation: lr_zip


#### Step 5: Error breakdown on validation (handbook §08)

In [9]:
val = data["val"]
pred = val_pred[chosen]

# price band: quintiles of the actual price
bands = pd.qcut(val[TARGET], 5, labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"])
error_by(val.assign(price_band=bands), pred, "price_band")

,n,MdAPE_%,MAPE_%,R2,MAE_$,within_10%
Q1 lowest,2283.0,9.42,15.37,-0.2287,63055.0,52.5
Q2,2293.0,7.99,11.64,-2.5581,78942.0,58.6
Q3,2272.0,8.38,11.57,-2.7238,103980.0,56.4
Q4,2284.0,9.49,12.55,-1.2190,165121.0,52.3
Q5 highest,2280.0,13.33,16.68,0.5020,494606.0,39.4


In [10]:
# by county, largest 10 counties in validation
top_counties = val["CountyOrParish"].value_counts().index[:10]
in_top = val["CountyOrParish"].isin(top_counties)
error_by(val[in_top], pred[in_top.to_numpy()], "CountyOrParish").sort_values("n", ascending=False)

,n,MdAPE_%,MAPE_%,R2,MAE_$,within_10%
Los Angeles,2785.0,10.00,13.90,0.8010,204626.0,50.0
Riverside,1917.0,9.37,14.54,0.5828,146008.0,52.6
San Diego,1273.0,9.35,12.77,0.8636,192505.0,52.1
San Bernardino,1196.0,7.84,11.77,0.7842,68574.0,59.8
Orange,1099.0,10.23,12.89,0.7962,242505.0,49.1
Contra Costa,538.0,8.54,12.46,0.8114,171932.0,55.0
Alameda,491.0,11.20,15.40,0.7323,228433.0,44.2
Ventura,369.0,8.55,13.93,0.7234,197971.0,54.2
Santa Clara,337.0,9.37,12.30,0.8434,303998.0,52.5
San Luis Obispo,202.0,11.89,14.45,0.7207,177502.0,45.5


In [11]:
# Model description: largest coefficients (on log price, scaled numeric inputs).
# A coefficient of 0.10 means roughly +10% price for +1 standard deviation (numeric) or for that category.
lr = models[chosen].regressor_
names = lr.named_steps["pre"].get_feature_names_out()
coefs = pd.Series(lr.named_steps["lr"].coef_, index=names)
numeric_coefs = coefs[coefs.index.str.startswith(("num__", "log__"))].sort_values()
numeric_coefs

num__Latitude                        -0.500947
num__Longitude                       -0.447109
num__Stories                         -0.032532
num__Stories_missing                 -0.024915
num__month_cos                       -0.009312
num__BedroomsTotal                   -0.008016
num__ViewYN_missing                  -0.005221
num__BedroomsTotal_missing           -0.002172
num__FireplaceYN_missing             -0.001870
num__BathroomsTotalInteger_missing   -0.001536
num__AttachedGarageYN                -0.000838
num__YearBuilt_missing               -0.000156
num__LotSizeSquareFeet_missing        0.000325
num__NewConstructionYN_missing        0.000811
num__ParkingTotal_missing             0.001344
num__GarageSpaces_missing             0.002490
num__month_sin                        0.003285
num__LivingArea_missing               0.004898
num__FireplaceYN                      0.004938
num__AttachedGarageYN_missing         0.007005
num__ParkingTotal                     0.011014
num__ViewYN  

#### Step 6: Save the model and results

In [12]:
# Everything the Week 9 app needs to rebuild a prediction: the fitted pipeline and the exact input columns
joblib.dump(models[chosen], os.path.join(MODEL_DIR, "baseline_lr.joblib"))
with open(os.path.join(MODEL_DIR, "baseline_lr_features.json"), "w") as f:
    json.dump({"model": chosen, "features": features, "categorical": variants[chosen], "logged": LOG_COLS,
               "seed": SEED}, f, indent=2)

val_results.to_csv(os.path.join(MODEL_DIR, "baseline_metrics.csv"))  # validation only
print("saved to", MODEL_DIR)

saved to dataset/processed/models


#### Findings
- **Baseline to beat (validation)**: `lr_zip`, MdAPE **9.52%**, MAPE 13.56%, R² 0.830, MAE $181k; 52% of homes within 10%. Test is scored once in Week 8, with all models together.
- **Dataset A + shared pipeline barely moved the baseline** (9.47% → 9.52% MdAPE). The fills now happen inside `idx_pipeline.py` instead of the CSV, and rows that used to be dropped (0 / outside-CA coordinates, the on-hold size-vs-rooms and price/sqft rules) are kept, so validation has 11,412 homes instead of 11,343.
- **Location does most of the work for a linear model**: adding ZIP cut MdAPE from 17.9% (`lr_county`) to 9.5%. Lat/long on their own are linear, and price doesn't change linearly across California, so ZIP one-hot does what lat/long can't.
- **No overfitting**: train and validation scores are within about 0.2 MdAPE points.
- **Weakest on the most expensive homes**: the top price quintile has MdAPE 13.3%, compared with 8–9.5% for the others. R² inside one price band is often negative because each band covers a narrow price range, so use MdAPE/MAPE to compare bands, not R².
- **Logging the sizes was necessary**: with raw LivingArea and lot size, a few 14–17k sqft homes were predicted at $50–127M, and validation R² was −0.99 even though MdAPE looked fine. That's why all four metrics get reported.
- **Coefficients**: log LivingArea dominates (+20% per standard deviation), then bathrooms and YearBuilt/AgeAtSale. YearBuilt and AgeAtSale are almost the same variable (AgeAtSale = sale year − YearBuilt), so their separate coefficients shouldn't be read on their own.
- **Data issues**: the Dataset A ranges fixed the 45-bedroom home and GarageSpaces up to 600 (now max 17 bedrooms, 20 garage spaces). Lot size has no agreed upper bound: 192 train lots are over 10M sq ft (~230 acres), max 340 billion. Logging limits the damage for LR; raise an upper bound with the team.
- **Next (Weeks 5–7)**: tree models in 04 (no need to log inputs, handle nonlinear location), then target encoding, geo and comps features in 05.